# 1-3. Qdrant 기반 Naive RAG

## 학습 목표

- PDF 청크를 임베딩하여 Qdrant에 저장한다.
- 사용자 질문과 유사한 문단을 검색한다.
- 검색 문단을 LLM 답변 생성에 연결한다.


## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼.pdf`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant 로컬 인메모리 모드

> 이 노트북은 `src` 파일을 import하지 않고, 노트북 안의 코드만으로 실행되도록 구성한다.


In [1]:
from pathlib import Path
import re
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(override=True, dotenv_path="../.env")

PDF_PATH = Path("../data/공직자_민원응대_핵심_매뉴얼.pdf")

/var/folders/cn/n1wp5rkx27j7415_2rb44sk00000gn/T/ipykernel_16000/959155110.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. PDF 로딩 + 청크 분할 함수


In [2]:
def clean_text(text: str) -> str:
    """PDF에서 추출한 텍스트를 기본 정제한다."""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_pdf_pages(pdf_path: Path) -> list[Document]:
    """PDF를 페이지 단위 Document 리스트로 로드한다."""
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    page_docs = []

    for doc in docs:
        text = clean_text(doc.page_content or "")
        if not text:
            continue

        page_docs.append(
            Document(
                page_content=text,
                metadata={
                    **doc.metadata,
                    "source": pdf_path.name,
                    "page": doc.metadata.get("page", 0) + 1,
                },
            )
        )
    return page_docs


def split_docs(docs: list[Document]) -> list[Document]:
    """페이지 단위 문서를 검색용 청크로 분할한다."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=120,
        separators=["\n\n", "\n", "STEP", "‣", "•", ". ", " "],
    )
    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i

    return chunks

## 2. 문서 청크 생성


In [3]:
pages = load_pdf_pages(PDF_PATH)
chunks = split_docs(pages)

print("페이지 수:", len(pages))
print("청크 수:", len(chunks))

페이지 수: 20
청크 수: 43


## 3. Qdrant 벡터스토어 생성

실습 편의를 위해 Qdrant를 인메모리 모드로 사용한다.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from const.const import create_model, create_embedding

embeddings = create_embedding()

vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",  # 인메모리 모드
    collection_name="civil_service_manual",
    force_recreate=True,
)

print("Qdrant 저장 완료")

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


Qdrant 저장 완료


## 4. 유사 문서 검색


In [5]:
question = "민원인이 반복 전화를 하면 어떻게 해야 하나요?"

retrieved_docs = vectorstore.similarity_search(question, k=4)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- 검색 결과 {i} ---")
    print(doc.metadata)
    print(doc.page_content[:600])

--- 검색 결과 1 ---
{'producer': 'Adobe PDF Library 11.0', 'creator': 'Adobe InDesign CC 2014 (Windows)', 'creationdate': '2025-11-30T19:46:56+09:00', 'moddate': '2025-11-30T19:47:04+09:00', 'trapped': '/False', 'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'total_pages': 20, 'page': 8, 'page_label': '8', 'chunk_id': 9, '_id': 'f536d460541c4dc9bbf91d9cd27e7eb4', '_collection_name': 'civil_service_manual'}
Ⅲ. 특이민원 응대요령 | 15
Ⅲ 특이민원 응대요령
Ⅲ. 특이민원 응대요령
01 공통사항
• 감정적 맞대응 금지, 침착하고 일관된 민원 응대 유지STEP
1
• 조언과 충고, 개인 견해 제시 또는 언쟁을 피하고 법령 및 규정에
 근거한 원칙에 따른 대응
STEP
2
• 위법행위 등에 대한 구체적 입증을 위한 증거자료 확보STEP
3
• 위법행위로부터 담당자를 보호하는 것을 우선하고 부서원과 상황을 
 공유하여 개인이 아닌 부서 차원에서 대응
STEP
4
02 전화응대
정당한 사유 없는 반복 전화 ※ 동일민원을 3회 이상 제기2-1
‣ 부서장 보고, 휴게시간 부여 등 
 반복전화를 받은 부서원에게 30분 이내 휴게시간 부여 가능
 ※ 통화 내용과 정도를 감안하여 필요시 위법행위 관리대장을 작성하여 채증
2단계
상황 보고
핵심대응 다른민원 처리를 위해 통화 지속 곤란 안내 후 종료
‣ 통화 지속 곤란을 설명한 후 상담종료 
 정당한 행정처분에 불복하여 반복전화하는 경우
 선생님, 말씀하신 민원은 정해진 규정과 절차에 따라 정상적으로 처리된 
 사항입니다. (또는 현재 처리 중에 있습니다.) 동일한 답변 외에는 도움을 
 드릴 수 없으므로 다른 민원처리를 위해 
--- 검색 

## 5. 검색 결과를 프롬프트에 넣기


In [6]:
def format_docs(docs: list[Document]) -> str:
    formatted = []

    for i, doc in enumerate(docs, start=1):
        formatted.append(
            f"[근거 {i}] page={doc.metadata.get('page')}, chunk={doc.metadata.get('chunk_id')}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted)


context = format_docs(retrieved_docs)
print(context[:1500])

[근거 1] page=8, chunk=9
Ⅲ. 특이민원 응대요령 | 15
Ⅲ 특이민원 응대요령
Ⅲ. 특이민원 응대요령
01 공통사항
• 감정적 맞대응 금지, 침착하고 일관된 민원 응대 유지STEP
1
• 조언과 충고, 개인 견해 제시 또는 언쟁을 피하고 법령 및 규정에
 근거한 원칙에 따른 대응
STEP
2
• 위법행위 등에 대한 구체적 입증을 위한 증거자료 확보STEP
3
• 위법행위로부터 담당자를 보호하는 것을 우선하고 부서원과 상황을 
 공유하여 개인이 아닌 부서 차원에서 대응
STEP
4
02 전화응대
정당한 사유 없는 반복 전화 ※ 동일민원을 3회 이상 제기2-1
‣ 부서장 보고, 휴게시간 부여 등 
 반복전화를 받은 부서원에게 30분 이내 휴게시간 부여 가능
 ※ 통화 내용과 정도를 감안하여 필요시 위법행위 관리대장을 작성하여 채증
2단계
상황 보고
핵심대응 다른민원 처리를 위해 통화 지속 곤란 안내 후 종료
‣ 통화 지속 곤란을 설명한 후 상담종료 
 정당한 행정처분에 불복하여 반복전화하는 경우
 선생님, 말씀하신 민원은 정해진 규정과 절차에 따라 정상적으로 처리된 
 사항입니다. (또는 현재 처리 중에 있습니다.) 동일한 답변 외에는 도움을 
 드릴 수 없으므로 다른 민원처리를 위해 통화를 종료하오니 양해하여 주시기 
 바랍니다. ※ 국민권익위원회 고충민원 등 신청 안내
 충분한 설명이 이루어졌음에도 반복전화하는 경우

[근거 2] page=8, chunk=10
드릴 수 없으므로 다른 민원처리를 위해 통화를 종료하오니 양해하여 주시기 
 바랍니다. ※ 국민권익위원회 고충민원 등 신청 안내
 충분한 설명이 이루어졌음에도 반복전화하는 경우
 선생님, 말씀하시는 내용에 대해서는 충분히 설명드렸습니다. 다른 분들의 
 민원처리가 지연되고 있어 통화를 종료하겠습니다.
 ※ 지속될 경우 행정기관에 부당한 요구를 하거나 반복적으로 전화하여 다른 
 민원처리를 지연시키는 행위는 공무방해행위에 해당됨을 경고
1단계
통화 곤란 안내 후
상담 종료

[근거 3] p

## 6. LLM 답변 생성


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from const.const import create_model, create_embedding

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 공직자 민원응대 매뉴얼 기반 업무지원 AI입니다.
            반드시 제공된 근거 문단 안의 내용만 사용해 답변하세요.
            근거에 없는 내용은 추측하지 마세요.

            답변 형식:
            1. 핵심 대응
            2. 단계별 조치
            3. 사용할 수 있는 안내 표현
            4. 근거
            5. 주의사항
            """,
        ),
        (
            "human",
            """
            질문:
            {question}

            근거 문단:
            {context}

            위 근거를 바탕으로 답변하세요.
            """,
        ),
    ]
)

llm = create_model()

rag_chain = prompt | llm | StrOutputParser()

answer = rag_chain.invoke({"question": question, "context": context})

print(answer)

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found z-ai/glm-5.3-flash in available_models, but type is unknown and inference may fail.
  warnings.warn(


## 핵심 정리

```python
question
→ vectorstore.similarity_search(question)
→ format_docs(retrieved_docs)
→ prompt + llm
→ answer
```

- Naive RAG는 질문을 그대로 검색한다.
- 검색 품질이 낮으면 답변 품질도 낮아진다.
- 다음 단계에서는 검색 품질을 개선하는 Advanced RAG를 다룬다.
